# Parkshare - Setup DB SQLite (sources brutes)


In [2]:
from pathlib import Path
import sqlite3
import pandas as pd

ROOT = Path.cwd().resolve().parent
SOURCES_DIR = ROOT / "data" / "Sources"
DB_DIR = Path.cwd() / "db"
DB_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = DB_DIR / "parkshare_raw.db"
SCHEMA_PATH = Path.cwd() / "sql" / "schema_raw.sql"

print("ROOT:", ROOT)
print("SOURCES_DIR:", SOURCES_DIR)
print("DB_PATH:", DB_PATH)
print("SCHEMA_PATH:", SCHEMA_PATH)


ROOT: C:\Users\liena\Travaux\B3 - Dev\challenge48h\parkshare_challenge48h_equipe9
SOURCES_DIR: C:\Users\liena\Travaux\B3 - Dev\challenge48h\parkshare_challenge48h_equipe9\data\Sources
DB_PATH: c:\Users\liena\Travaux\B3 - Dev\challenge48h\parkshare_challenge48h_equipe9\app\db\parkshare_raw.db
SCHEMA_PATH: c:\Users\liena\Travaux\B3 - Dev\challenge48h\parkshare_challenge48h_equipe9\app\sql\schema_raw.sql


In [3]:
files_config = {
    "AAA_OPENDATA_RECHARGEABLE_2025-11-24.csv": {
        "table": "raw_aaa_opendata_rechargeable",
        "sep": ",",
        "encoding": "latin1",
    },
    "tableau-synthetique-coproff-com-2024.csv": {
        "table": "raw_tableau_synthetique_coproff",
        "sep": ";",
        "encoding": "utf-8",
    },
    "population_INSEE.csv": {
        "table": "raw_population_insee",
        "sep": ",",
        "encoding": "utf-8",
    },
    "Mapping.csv": {
        "table": "raw_mapping",
        "sep": ";",
        "encoding": "utf-8",
    },
    "rnc-data-gouv-with-qpv.csv": {
        "table": "raw_rnc_data_gouv_with_qpv",
        "sep": ",",
        "encoding": "utf-8",
    },
}

files_config

{'AAA_OPENDATA_RECHARGEABLE_2025-11-24.csv': {'table': 'raw_aaa_opendata_rechargeable',
  'sep': ',',
  'encoding': 'latin1'},
 'tableau-synthetique-coproff-com-2024.csv': {'table': 'raw_tableau_synthetique_coproff',
  'sep': ';',
  'encoding': 'utf-8'},
 'population_INSEE.csv': {'table': 'raw_population_insee',
  'sep': ',',
  'encoding': 'utf-8'},
 'Mapping.csv': {'table': 'raw_mapping', 'sep': ';', 'encoding': 'utf-8'}}

## Inspection rapide (50 premieres lignes)

In [4]:
for file_name, cfg in files_config.items():
    path = SOURCES_DIR / file_name
    print("\n" + "=" * 80)
    print(f"Fichier: {file_name}")
    df_preview = pd.read_csv(
        path,
        sep=cfg["sep"],
        encoding=cfg["encoding"],
        dtype=str,
        nrows=50,
    )
    print(f"Colonnes ({len(df_preview.columns)}):", list(df_preview.columns))
    display(df_preview.head(5))



Fichier: AAA_OPENDATA_RECHARGEABLE_2025-11-24.csv
Colonnes (8): ['CODGEO', 'LIBGEO', 'EPCI', 'LIBEPCI', 'DATE_ARRETE', 'NB_VP_RECHARGEABLES_EL', 'NB_VP_RECHARGEABLES_GAZ', 'NB_VP']


,CODGEO,LIBGEO,EPCI,LIBEPCI,DATE_ARRETE,NB_VP_RECHARGEABLES_EL,NB_VP_RECHARGEABLES_GAZ,NB_VP
0,01022,ARTEMARE,200040350,CC Bugey Sud,2025-09-30,26,0,1446
1,01022,ARTEMARE,200040350,CC Bugey Sud,2025-06-30,27,0,1437
2,01022,ARTEMARE,200040350,CC Bugey Sud,2025-03-31,20,0,1425
3,01022,ARTEMARE,200040350,CC Bugey Sud,2024-12-31,20,0,1416
4,01022,ARTEMARE,200040350,CC Bugey Sud,2024-09-30,21,0,1406



Fichier: tableau-synthetique-coproff-com-2024.csv
Colonnes (22): ['nom_reg', 'code_reg', 'nom_dep', 'code_dep', 'code_epci', 'nom_epci', 'code_com', 'nom_commune', 'nb_copros', 'nb_logements', 'copros_5_moins', 'copros_6_10', 'copros_11_20', 'copros_21_50', 'copros_51_200', 'copros_200_plus', 'copros_avant_49', 'copros_49_74', 'copros_75_2000', 'copros_2000_plus', 'copros_annee_na', 'taux_immat']


,nom_reg,code_reg,nom_dep,code_dep,code_epci,nom_epci,code_com,nom_commune,nb_copros,nb_logements,...,copros_11_20,copros_21_50,copros_51_200,copros_200_plus,copros_avant_49,copros_49_74,copros_75_2000,copros_2000_plus,copros_annee_na,taux_immat
0,Auvergne-Rhône-Alpes,84,Ain,01,200069193,CC de la Dombes,01001,L'Abergement-Clémenciat,2,5,...,0,0,0,0,2,0,0,0,0,0.0
1,Auvergne-Rhône-Alpes,84,Ain,01,240100883,CC de la Plaine de l'Ain,01002,L'Abergement-de-Varey,1,2,...,0,0,0,0,1,0,0,0,0,0.0
2,Auvergne-Rhône-Alpes,84,Ain,01,240100883,CC de la Plaine de l'Ain,01004,Ambérieu-en-Bugey,268,2360,...,27,30,2,0,101,40,37,88,2,0.6754
3,Auvergne-Rhône-Alpes,84,Ain,01,200042497,CC Dombes Saône Vallée,01005,Ambérieux-en-Dombes,19,142,...,4,1,0,0,11,0,1,7,0,0.6316
4,Auvergne-Rhône-Alpes,84,Ain,01,200040350,CC Bugey Sud,01006,Ambléon,2,4,...,0,0,0,0,2,0,0,0,0,0.5



Fichier: population_INSEE.csv
Colonnes (23): ['code_insee', 'population_totale', 'pop_15_29', 'pop_60_74', 'pop_75_89', 'pop_90_plus', 'residences_principales', 'logements_vacants', 'logements_locatifs', 'pop_sans_diplome', 'pop_bac', 'pop_bac_plus_2', 'pop_bac_plus_5', 'pop_active', 'pop_chomeurs', 'pop_inactifs', 'revenu_median', 'taux_pauvrete', 'nb_bénéficiaires_rsa', 'nb_allocataires_minimum_vieillesse', 'nb_allocataires_famille', 'nb_minima_sociaux', 'nb_aides_logement']


,code_insee,population_totale,pop_15_29,pop_60_74,pop_75_89,pop_90_plus,residences_principales,logements_vacants,logements_locatifs,pop_sans_diplome,...,pop_active,pop_chomeurs,pop_inactifs,revenu_median,taux_pauvrete,nb_bénéficiaires_rsa,nb_allocataires_minimum_vieillesse,nb_allocataires_famille,nb_minima_sociaux,nb_aides_logement
0,01001,832.0,100.025669125929,149.720217782771,67.0983888134989,5.74532353065543,341.234562367113,17.4456416337286,47.1325549002898,97.2302110876128,...,417.591741457043,27.1665759093211,90.9204667840635,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,01002,267.0,32.4390934255431,40.5047400920238,19.768433507588,1.867717290097,115.722510420668,14.2557426112982,11.3851958208619,18.3130582188961,...,130.2251186101,2.14768904740552,32.806714778954,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,01004,14854.0,3081.37526584419,2269.03322839783,1152.13770206965,188.937108902086,6932.83919790318,775.203247637224,3913.53469957631,2277.96533299744,...,7118.70900322168,969.687159377275,2236.08253427179,50.0,17.0,3.4,25.3,2.6,3.0,1.4
3,01005,1897.0,289.213772744191,301.217071564669,123.672439577029,11.0792647363145,794.75302490539,96.1291989664086,199.092413871919,250.294831905072,...,987.970116130784,62.2094464439419,227.339764402559,61.0,NaN,4.3,21.4,2.1,0.9,0.4
4,01006,113.0,13.8771929824561,37.6666666666667,7.92982456140351,0.0,56.5,7.08333333333333,14.8684210526316,19.8245614035088,...,50.5526315789474,4.95614035087719,20.8157894736842,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Fichier: Mapping.csv
Colonnes (4): ['code_insee', 'code_postal', 'nom_commune', 'departement']


,code_insee,code_postal,nom_commune,departement
0,01001,1400,L ABERGEMENT CLEMENCIAT,01
1,01002,1640,L ABERGEMENT DE VAREY,01
2,01004,1500,AMBERIEU EN BUGEY,01
3,01005,1330,AMBERIEUX EN DOMBES,01
4,01006,1300,AMBLEON,01


## Creation du schema brut

In [5]:
assert SCHEMA_PATH.exists(), f"Schema SQL introuvable: {SCHEMA_PATH}"

with sqlite3.connect(DB_PATH) as conn:
    schema_sql = SCHEMA_PATH.read_text(encoding="utf-8")
    conn.executescript(schema_sql)
    print("Schema cree avec succes.")


Schema cree avec succes.


## Chargement echantillon (1000 lignes / fichier)

Aucune transformation metier. Toutes les colonnes sont chargees en texte pour conserver le brut.

In [6]:
N_SAMPLE = 1000

with sqlite3.connect(DB_PATH) as conn:
    for file_name, cfg in files_config.items():
        path = SOURCES_DIR / file_name
        table_name = cfg["table"]

        df = pd.read_csv(
            path,
            sep=cfg["sep"],
            encoding=cfg["encoding"],
            dtype=str,
            nrows=N_SAMPLE,
        )

        # Standardisation minimale de nom de colonne pour compatibilite SQL.
        # Le contenu des valeurs reste strictement brut.
        df.columns = [
            c.strip()
            .replace(" ", "_")
            .replace("-", "_")
            .replace("%", "pct")
            .replace("é", "e")
            .replace("è", "e")
            .replace("ê", "e")
            .replace("à", "a")
            .replace("û", "u")
            .replace("ô", "o")
            .replace("ç", "c")
            for c in df.columns
        ]

        df["source_file"] = file_name

        df.to_sql(table_name, conn, if_exists="append", index=False)
        print(f"{table_name}: {len(df)} lignes chargees.")


raw_aaa_opendata_rechargeable: 1000 lignes chargees.
raw_tableau_synthetique_coproff: 1000 lignes chargees.
raw_population_insee: 1000 lignes chargees.
raw_mapping: 1000 lignes chargees.


## Verification rapide

In [ ]:
tables = [
    "raw_aaa_opendata_rechargeable",
    "raw_tableau_synthetique_coproff",
    "raw_population_insee",
    "raw_mapping",
    "raw_rnc_data_gouv_with_qpv",
]

with sqlite3.connect(DB_PATH) as conn:
    for t in tables:
        count = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
        print(f"{t}: {count} lignes")
